# Mô hình Baseline: SARIMAX
Trong phần này, chúng ta sẽ xây dựng mô hình thống kê SARIMAX làm cơ sở (baseline).
Vì SARIMAX hoạt động tốt với dữ liệu có tính chu kỳ cố định và đơn giản, chúng ta sẽ chuyển đổi dữ liệu theo giờ (hourly) thành dữ liệu trung bình ngày (daily) để tránh mô hình bị nhiễu bởi chu kỳ phức tạp trong ngày.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# Đọc dữ liệu đúng file đã làm sạch
df = pd.read_csv('../data/processed/pm25_training_data.csv')
df['datetime'] = pd.to_datetime(df['datetime'])

# Chọn 1 thành phố đại diện (Hà Nội - nơi có ô nhiễm rõ rệt nhất)
df_hn = df[df['city'] == 'Hà Nội'].sort_values('datetime').set_index('datetime')

# Resample về trung bình ngày (Daily)
df_daily = df_hn.resample('D').mean(numeric_only=True)

# Bỏ các ngày bị thiếu PM2.5 (nếu có)
df_daily = df_daily.dropna(subset=['pm25'])

print(f"Dữ liệu sau khi gom theo ngày còn lại: {len(df_daily)} ngày")
display(df_daily.head(3))

## Chia tập Train/Test
Chúng ta sử dụng **Time-series split** (chia theo trình tự thời gian). 
Giữ lại 80% dữ liệu ở quá khứ để huấn luyện và 20% dữ liệu ở tương lai (gần nhất) để kiểm thử. Không được phép chia ngẫu nhiên (random split) đối với chuỗi thời gian.

In [ ]:
# Cắt 80% thời gian đầu làm Train, 20% thời gian sau làm Test
train_size = int(len(df_daily) * 0.8)
train = df_daily.iloc[:train_size]
test = df_daily.iloc[train_size:]

print(f"Tập Train: Từ {train.index.min().date()} đến {train.index.max().date()} ({len(train)} ngày)")
print(f"Tập Test:  Từ {test.index.min().date()} đến {test.index.max().date()} ({len(test)} ngày)")

# Chọn biến mục tiêu (Target) và biến ngoại sinh (Exogenous)
target = 'pm25'
exog_cols = ['temp', 'humidity', 'wind_speed', 'pressure']

y_train, X_train = train[target], train[exog_cols]
y_test, X_test = test[target], test[exog_cols]

## Huấn luyện và Đánh giá SARIMAX
Sử dụng mô hình SARIMAX với các tham số dự đoán mùa vụ và kết hợp thêm các biến ngoại sinh.

In [ ]:
print("Đang huấn luyện SARIMAX (quá trình này có thể mất vài phút)...")

# Cấu hình SARIMAX(p,d,q)(P,D,Q,s)
# s=7 để mô hình hiểu chu kỳ 7 ngày trong tuần
model = SARIMAX(
    y_train,
    exog=X_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7)
)
results = model.fit(disp=False)

# Dự đoán trên tập Test
predictions = results.predict(
    start=len(train),
    end=len(train) + len(test) - 1,
    exog=X_test
)

# Tính toán sai số
rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)

print(f"\n📊 KẾT QUẢ SARIMAX BASELINE:")
print(f"   RMSE: {rmse:.2f} µg/m³")
print(f"   MAE:  {mae:.2f} µg/m³")

# Trực quan hóa kết quả
plt.figure(figsize=(14, 5))
plt.plot(y_test.index, y_test.values, label='Thực tế (PM2.5)', alpha=0.7)
plt.plot(y_test.index, predictions.values, color='red', label='Dự đoán SARIMAX', alpha=0.8)
plt.title('SARIMAX Baseline: Dự báo PM2.5 tại Hà Nội')
plt.ylabel('PM2.5 (µg/m³)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()